# Lesson 10.7 — Production MCP Patterns
Netsetos GenAI Course v2.0 | Module 10: Tool Use & MCP

5 production pillars: rate limit, error recovery, logging, observability, security. Plus auth, multi-tenancy, versioning, health checks.


In [ ]:
print('Module 10: Tool Use & MCP')
print('Lesson 10.7: Production MCP Patterns')


## Ex 1: The Five Production Pillars


In [ ]:
print('Five pillars every production MCP server needs:')
for pillar, why, common_tool in [
    ('Rate limiting',     'one noisy client cannot starve others', 'token bucket per client_id'),
    ('Error recovery',     'transient failures must not cascade',  '3-state breaker per tool + retry'),
    ('Structured logging','queryable JSON, not free-text',         'JSON + W3C trace_id'),
    ('Observability',     'know which tool, which tenant, which p99', 'OTel + Prometheus + Grafana'),
    ('Security',          'auth + tool allowlist + tenant isolation', 'OAuth 2.0 + scoped tokens'),
]:
    print(f'  {pillar:18s}: {why:42s} | tool: {common_tool}')


## Ex 2: Per-Client Token Bucket


In [ ]:
print('Token bucket parameters by tier (illustrative):')
for tier, capacity, refill_per_sec, burst in [
    ('free',       10,   1, '10 in burst, 1/sec sustained'),
    ('pro',       100,  10, '100 burst, 10/sec'),
    ('enterprise',1000,100, '1000 burst, 100/sec'),
]:
    print(f'  {tier:12s}: capacity={capacity:>4} refill={refill_per_sec:>3}/s | {burst}')
print()
print('Empty bucket -> 429 + Retry-After: ceil(1/refill) seconds.')


## Ex 3: Per-Tool Circuit Breaker


In [ ]:
print('Per-tool 3-state breaker:')
for state, behavior in [
    ('CLOSED',     'normal: tool calls pass through'),
    ('OPEN',       'reject instantly with status=circuit_open; LLM decides to retry/skip'),
    ('HALF_OPEN',  'one probe call after cooldown; success -> CLOSED, fail -> OPEN'),
]:
    print(f'  {state:10s}: {behavior}')
print()
print('Trip per-TOOL not per-server: one flaky tool does not down the whole MCP.')


## Ex 4: Structured Error Result


In [ ]:
print('Tool error result the LLM can reason about (NOT a stack trace):')
import json
tool_error = {
    'status': 'error',
    'error_kind': 'rate_limited',
    'message': 'Per-tool rate limit; retry in 5s.',
    'retry_after_seconds': 5,
    'suggested_action': 'wait + retry, or skip this tool',
    'trace_id': '7f3a-...',
}
print(json.dumps(tool_error, indent=2))
print()
print('Server logs the stack trace separately; LLM sees actionable JSON.')


## Ex 5: Structured JSON Log


In [ ]:
print('Per-call JSON log line:')
import json
log = {
    'ts': '2026-04-18T10:42:15Z',
    'event': 'tool_call',
    'tenant': 'acme',
    'client_id': 'usr_042',
    'tool': 'web_search',
    'latency_ms': 412,
    'status': 'ok',
    'trace_id': '7f3a-...',
    'parent_span': 'b210-...',
}
print(json.dumps(log))
print()
print('Query in BigQuery / Elasticsearch / Loki by any field. Never free-text logs.')


## Ex 6: Health Check Tiers


In [ ]:
print('Three health-check tiers for MCP:')
for tier, checks, k8s_use in [
    ('liveness',  'process responsive (HTTP 200 on /healthz)',     'restarts pod on fail'),
    ('readiness', 'deps reachable (DB ping, downstream API)',      'drains traffic on fail'),
    ('tool-level','per-tool last-success ts + success rate',        'feeds breaker + dashboards'),
]:
    print(f'  {tier:10s}: {checks:48s} | k8s: {k8s_use}')


## Ex 7: Auth + Multi-Tenancy Checklist


In [ ]:
print('Production auth + multi-tenant checklist:')
for n, item in enumerate([
    'OAuth 2.0 for user-context tools; API key for service-to-service; mTLS for zero-trust',
    'Scoped tokens (per-tool allowlist baked into token claims)',
    'Short-lived tokens (1h-1d), rotation supported',
    'Per-tenant rate limit + per-tenant audit log',
    'Tool implementations MUST filter by tenant_id (never trust)',
    'tenant_id propagated through every span + log',
    'Cross-tenant leakage is a SEV1; nightly isolation tests (12.4)',
], 1):
    print(f'  {n}. {item}')


---
## Recap
5 pillars: rate limit + error recovery + structured logging + observability + security.
Per-CLIENT token bucket; per-TOOL circuit breaker; per-TENANT isolation.
Structured JSON errors the LLM can reason about; stack traces stay server-side.
Semver tool names with 30-90d deprecation. Health: liveness + readiness + per-tool.
End of Module 10.
